# Fraud Shield - Baseline Supervised Models

Trains and evaluates the three supervised baselines -- Logistic Regression,
Random Forest, and XGBoost -- using the engineered features from
`02_feature_engineering.ipynb`. These become the supervised branch that
feeds the hybrid ensemble in `05_hybrid_ensemble.ipynb`.

**Inputs:** `data/processed/train_features.parquet`, `data/processed/test_features.parquet`
**Outputs:** trained models saved to `data/processed/models/`, validation metrics


In [ ]:
import sys
sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.models.supervised import (
    train_logistic_regression,
    train_random_forest,
    train_xgboost,
)
from src.models.evaluate import compute_metrics, print_report, explain_model

MODELS_DIR = Path('../data/processed/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load engineered features


In [ ]:
train_df = pd.read_parquet('../data/processed/train_features.parquet')
test_df = pd.read_parquet('../data/processed/test_features.parquet')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()


## 2. Split features/target

`transaction_id`, `event_time`, and `cc_num` are identifiers, not model
inputs -- keep them aside for traceability but drop them from `X`.


In [ ]:
id_cols = ['transaction_id', 'event_time', 'cc_num']
target_col = 'is_fraud'
feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

X = train_df[feature_cols]
y = train_df[target_col]

print(f'{len(feature_cols)} features:')
print(feature_cols)


## 3. Train / validation split

`fraudTest.csv` is the true held-out test set (used only at the very end).
Here we carve a validation split out of the training data for model
selection and comparison, stratified on `is_fraud` to preserve the fraud rate
in both splits.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}, fraud rate {y_train.mean()*100:.4f}%')
print(f'Val:   {X_val.shape}, fraud rate {y_val.mean()*100:.4f}%')


## 4. Class imbalance handling

`train_logistic_regression` and `train_random_forest` already use
`class_weight='balanced'` by default. XGBoost needs an explicit
`scale_pos_weight` (ratio of negative to positive class counts).


In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f'Negative: {neg}, Positive: {pos}, scale_pos_weight: {scale_pos_weight:.2f}')


## 5. Train Logistic Regression


In [ ]:
logreg = train_logistic_regression(X_train, y_train)

logreg_pred = logreg.predict(X_val)
logreg_proba = logreg.predict_proba(X_val)[:, 1]

logreg_metrics = compute_metrics(y_val, logreg_pred, logreg_proba)
print(logreg_metrics)
print_report(y_val, logreg_pred)


## 6. Train Random Forest


In [ ]:
rf = train_random_forest(X_train, y_train)

rf_pred = rf.predict(X_val)
rf_proba = rf.predict_proba(X_val)[:, 1]

rf_metrics = compute_metrics(y_val, rf_pred, rf_proba)
print(rf_metrics)
print_report(y_val, rf_pred)


## 7. Train XGBoost


In [ ]:
xgb_model = train_xgboost(X_train, y_train, scale_pos_weight=scale_pos_weight)

xgb_pred = xgb_model.predict(X_val)
xgb_proba = xgb_model.predict_proba(X_val)[:, 1]

xgb_metrics = compute_metrics(y_val, xgb_pred, xgb_proba)
print(xgb_metrics)
print_report(y_val, xgb_pred)


## 8. Compare models


In [ ]:
results = pd.DataFrame({
    'Logistic Regression': logreg_metrics,
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
}).T

results = results.sort_values('f1', ascending=False)
results


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
results[['precision', 'recall', 'f1', 'auc_roc']].plot(kind='bar', ax=ax)
ax.set_title('Baseline model comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Feature importance / interpretability (XGBoost)

XGBoost typically performs best on tabular fraud data and pairs cleanly
with SHAP's TreeExplainer -- this doubles as the interpretability
deliverable the rubric calls for.


In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='teal')
ax.invert_yaxis()
ax.set_title('XGBoost - top 15 feature importances')
plt.tight_layout()
plt.show()


In [ ]:
# SHAP can be slow on the full validation set -- sample for a readable summary plot
X_val_sample = X_val.sample(n=min(2000, len(X_val)), random_state=42)
shap_values = explain_model(xgb_model, X_val_sample)

import shap
shap.summary_plot(shap_values, X_val_sample, show=True)


## 10. Save trained models

Saved under `data/processed/models/` (gitignored, same as the rest of
`data/`) so they're available to `05_hybrid_ensemble.ipynb` without
retraining.


In [ ]:
joblib.dump(logreg, MODELS_DIR / 'logreg.joblib')
joblib.dump(rf, MODELS_DIR / 'random_forest.joblib')
joblib.dump(xgb_model, MODELS_DIR / 'xgboost.joblib')

# Also persist validation predictions -- reused as meta-features when building
# the hybrid ensemble, without needing to retrain here
val_predictions = pd.DataFrame({
    'is_fraud': y_val.values,
    'logreg_proba': logreg_proba,
    'rf_proba': rf_proba,
    'xgb_proba': xgb_proba,
})
val_predictions.to_parquet(MODELS_DIR / 'baseline_val_predictions.parquet', index=False)

print('Saved models and validation predictions to', MODELS_DIR)


## 11. Next steps

- Compare against `04_deep_learning_models.ipynb` (FNN / LSTM branch)
- Feed `baseline_val_predictions.parquet` + the deep learning branch's
  validation probabilities into `05_hybrid_ensemble.ipynb`
- Note any weak spots here (e.g. recall on the fraud class) to target
  specifically when tuning the deep learning models
